# Representations

Once periods are grouped into clusters ([Clustering methods](clustering_methods.ipynb)),
each cluster must be condensed into a **single representative profile**. The
representation controls *what* that profile preserves — and the choice matters more
than people expect.

Set it with `ClusterConfig(representation=...)`:

| representation | the profile is… | preserves |
|----------------|-----------------|-----------|
| `medoid` (default) | a real period from the cluster | realistic shape |
| `mean` | the average of the cluster | smooth, central tendency |
| `maxoid` | the most extreme period | peaks |
| `distribution` | reshaped to match the value histogram | the **duration curve** |
| `distribution_minmax` | distribution + exact per-step min/max | distribution **and** extremes |
| `minmax_mean` | mean, but min/max kept per time step | extremes around the average |

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## They score differently on different things

Two errors tell different stories: plain **RMSE** measures point-by-point timing,
while **RMSE on the duration curve** measures how well the *spread* of values is
kept (peaks, valleys, how often each level occurs). A representation can win one
and lose the other:

In [ ]:
from tsam import ClusterConfig

reps = [
    "medoid",
    "mean",
    "maxoid",
    "distribution",
    "distribution_minmax",
    "minmax_mean",
]
rows = {}
for rep in reps:
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        cluster=ClusterConfig(representation=rep),
    )
    rows[rep] = {
        "RMSE": round(float(r.accuracy.rmse.mean()), 4),
        "RMSE (duration curve)": round(float(r.accuracy.rmse_duration.mean()), 4),
    }
pd.DataFrame(rows).T

## Seeing it on the duration curve

The duration curve sorts every value high-to-low, so it ignores *when* things
happen and shows the *distribution* of values. `mean` and `medoid` shave the peaks
and fill the valleys; `distribution` is built to trace this curve almost exactly:

In [ ]:
frames = []
for rep in ["medoid", "mean", "distribution", "minmax_mean"]:
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        cluster=ClusterConfig(representation=rep),
    )
    s = r.reconstructed["Load"].sort_values(ascending=False).reset_index(drop=True)
    frames.append(
        pd.DataFrame({"rank": range(len(s)), "Load": s.values, "representation": rep})
    )
orig = data["Load"].sort_values(ascending=False).reset_index(drop=True)
frames.append(
    pd.DataFrame(
        {"rank": range(len(orig)), "Load": orig.values, "representation": "original"}
    )
)
px.line(
    pd.concat(frames),
    x="rank",
    y="Load",
    color="representation",
    title="Duration curve by representation",
)

## Why `distribution` matters for storage and peaks

If a downstream model sizes storage or charges peak-based fees, getting the
*distribution* right matters more than the exact timing. Note how `distribution`
slashes the duration-curve error while costing little in timing RMSE — and how
`mean` quietly clips the maximum:

In [ ]:
mean_rep = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    cluster=ClusterConfig(representation="mean"),
)
dist_rep = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    cluster=ClusterConfig(representation="distribution"),
)
pd.DataFrame(
    {
        "original": [data["Load"].max(), data["Load"].min()],
        "mean": [
            mean_rep.reconstructed["Load"].max(),
            mean_rep.reconstructed["Load"].min(),
        ],
        "distribution": [
            dist_rep.reconstructed["Load"].max(),
            dist_rep.reconstructed["Load"].min(),
        ],
    },
    index=["peak Load", "min Load"],
).round(1)

## Which to use

- **`medoid`** (default) or **`mean`** for general use.
- **`distribution`** when the value distribution / duration curve drives the result
  (storage, peak pricing).
- **`minmax_mean`** or **`distribution_minmax`** when exact peaks must survive.

To force a *specific* extreme day to be kept verbatim, see
[Extreme periods](extreme_periods.ipynb).